# Exercise 2 — Code Generation with ReACT Prompting

**Course:** Applied AI — Prompt Engineering
**Goal:** Use a **ReACT** style (Reason + Act) approach to generate Python code, **run it**,
observe the result, and **fix** it — then show the working output.

## Tools used
- **Google Colab** (Python 3 runtime).
- **OpenAI API** (`gpt-4o-mini`) as the code-generating LLM, with a deterministic offline
  fallback so the notebook runs and shows output without a key.
- Python's built-in `exec()` to **actually execute** the model's generated code inside the
  notebook — the "Act / Observe" part of ReACT is real, not simulated.

## ReACT loop implemented here
```
Thought   -> reason about the task and plan the approach
Action    -> generate Python code
Observe   -> RUN the code + a test; capture result or error
Thought   -> diagnose what went wrong
Action    -> generate corrected code
Observe   -> RUN again; test passes -> Final Answer
```

## Task specification (the constraints given to the model)
- **Input:** an integer `n >= 0`.
- **Output:** a function `primes_up_to(n)` returning a sorted `list[int]` of all primes `<= n`.
- **Libraries:** standard library only (no imports required).
- **Formatting:** a single self-contained function; deterministic output.
- **Error handling:** `n < 2` must return `[]` (not crash).
- **Acceptance test:** `primes_up_to(10) == [2, 3, 5, 7]` and `primes_up_to(1) == []`.


In [1]:
# --- Setup: code-generating LLM with real OpenAI call + offline fallback ---
# In Colab, uncomment to install the SDK:
# !pip install openai
import os, io, contextlib, traceback

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = (
    "You are a senior Python engineer using the ReACT method. "
    "Follow strictly: think step by step (Thought), then output ONE Python code block (Action). "
    "When told an observation (a test failure/error), reason about the cause, then output a "
    "corrected code block. Standard library only. Return runnable code, no explanations "
    "outside the required Thought lines."
)

# Deterministic fallback code returned for the two ReACT rounds when offline.
# ROUND 1 contains a real, intentional bug (treats 1 as prime AND excludes n itself);
# ROUND 2 is the corrected version. This drives a genuine observe->fix cycle.
_FALLBACK_CODE_V1 = "\n".join([
    "def primes_up_to(n):",
    "    primes = []",
    "    for num in range(1, n):          # BUG 1: starts at 1; BUG 2: range excludes n",
    "        is_prime = True",
    "        for d in range(2, num):",
    "            if num % d == 0:",
    "                is_prime = False",
    "                break",
    "        if is_prime:",
    "            primes.append(num)",
    "    return primes",
])

_FALLBACK_CODE_V2 = "\n".join([
    "def primes_up_to(n):",
    "    if n < 2:                        # error handling: no primes below 2",
    "        return []",
    "    primes = []",
    "    for num in range(2, n + 1):      # start at 2, include n",
    "        is_prime = True",
    "        d = 2",
    "        while d * d <= num:          # only test up to sqrt(num)",
    "            if num % d == 0:",
    "                is_prime = False",
    "                break",
    "            d += 1",
    "        if is_prime:",
    "            primes.append(num)",
    "    return primes",
])

def generate_code(messages, round_tag):
    """Ask the LLM for code. Offline -> deterministic buggy-then-fixed code."""
    if USE_OPENAI:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(model=MODEL, temperature=0.1, messages=messages)
        text = resp.choices[0].message.content
        # extract the python code block
        if "```" in text:
            block = text.split("```")[1]
            block = block[len("python"):] if block.lstrip().startswith("python") else block
            return block.strip()
        return text.strip()
    return _FALLBACK_CODE_V1 if round_tag == "v1" else _FALLBACK_CODE_V2

print("Codegen backend:", "OpenAI " + MODEL if USE_OPENAI else "offline deterministic fallback")


Codegen backend: offline deterministic fallback


## The Act / Observe engine

`run_and_test` executes generated code in a fresh namespace and runs the acceptance test.
It returns a structured **observation** — exactly what a ReACT agent feeds back into its
next reasoning step.


In [2]:
def run_and_test(code):
    """Execute generated code + acceptance test. Return an observation dict."""
    ns = {}
    stdout = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, ns)                      # ACT: run the generated code
            f = ns["primes_up_to"]
            got10, got1 = f(10), f(1)           # OBSERVE: run the acceptance test
            assert got10 == [2, 3, 5, 7], f"primes_up_to(10) -> {got10}, expected [2, 3, 5, 7]"
            assert got1 == [], f"primes_up_to(1) -> {got1}, expected []"
        return {"passed": True, "detail": "All acceptance tests passed.",
                "sample": {"primes_up_to(10)": f(10)}}
    except Exception as e:
        return {"passed": False, "detail": f"{type(e).__name__}: {e}",
                "trace": traceback.format_exc(limit=1)}


## ReACT round 1 — reason, generate, run, observe

In [3]:
USER_TASK = (
    "Thought: We need primes_up_to(n) returning all primes <= n as a sorted list, "
    "standard library only, returning [] for n < 2.\n"
    "Action: Write the function."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_TASK},
]

print("THOUGHT (plan): iterate candidates, test primality, collect primes, handle n<2.\n")

code_v1 = generate_code(messages, round_tag="v1")
print("ACTION (generated code v1):\n")
print(code_v1)

obs1 = run_and_test(code_v1)
print("\nOBSERVATION:", "PASS" if obs1["passed"] else "FAIL")
print(obs1["detail"])


THOUGHT (plan): iterate candidates, test primality, collect primes, handle n<2.

ACTION (generated code v1):

def primes_up_to(n):
    primes = []
    for num in range(1, n):          # BUG 1: starts at 1; BUG 2: range excludes n
        is_prime = True
        for d in range(2, num):
            if num % d == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes

OBSERVATION: FAIL
AssertionError: primes_up_to(10) -> [1, 2, 3, 5, 7], expected [2, 3, 5, 7]


## ReACT round 2 — diagnose the failure, fix, re-run

We feed the **observation back to the model** as the next turn (this is the "acting on the
environment then reasoning again" loop). Offline, the fallback returns the corrected code.


In [4]:
# Feed the failed observation back into the conversation (ReACT feedback loop)
messages.append({"role": "assistant", "content": f"```python\n{code_v1}\n```"})
messages.append({"role": "user", "content": (
    f"Observation: the acceptance test FAILED -> {obs1['detail']}\n"
    "Thought: reason about why, then Action: return corrected code."
)})

print("THOUGHT (diagnosis): v1 started at 1 (1 is not prime) and used range(1, n), "
      "which both includes 1 and excludes n. Fix: start at 2, use range(2, n+1), "
      "add n<2 guard, and test divisors only up to sqrt(num).\n")

code_v2 = generate_code(messages, round_tag="v2")
print("ACTION (generated code v2):\n")
print(code_v2)

obs2 = run_and_test(code_v2)
print("\nOBSERVATION:", "PASS" if obs2["passed"] else "FAIL")
print(obs2["detail"])


THOUGHT (diagnosis): v1 started at 1 (1 is not prime) and used range(1, n), which both includes 1 and excludes n. Fix: start at 2, use range(2, n+1), add n<2 guard, and test divisors only up to sqrt(num).

ACTION (generated code v2):

def primes_up_to(n):
    if n < 2:                        # error handling: no primes below 2
        return []
    primes = []
    for num in range(2, n + 1):      # start at 2, include n
        is_prime = True
        d = 2
        while d * d <= num:          # only test up to sqrt(num)
            if num % d == 0:
                is_prime = False
                break
            d += 1
        if is_prime:
            primes.append(num)
    return primes

OBSERVATION: PASS
All acceptance tests passed.


## Final answer — use the corrected function

In [5]:
assert obs2["passed"], "Expected the corrected code to pass."

# Bring the validated function into the notebook namespace and use it.
exec(code_v2, globals())

primes_50 = primes_up_to(50)
print("FINAL ANSWER")
print("Primes up to 50:", primes_50)
print("Count:", len(primes_50))
print("Sum:", sum(primes_50))
print("Edge cases -> primes_up_to(0):", primes_up_to(0), "| primes_up_to(1):", primes_up_to(1))


FINAL ANSWER
Primes up to 50: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Count: 15
Sum: 328
Edge cases -> primes_up_to(0): [] | primes_up_to(1): []


## Notes on fixes / iteration

- **Round 1 bug (observed by running, not by guessing):** `range(1, n)` both included `1`
  (not prime) and excluded `n` itself, so `primes_up_to(10)` returned `[1, 2, 3, 5, 7]`.
  The acceptance test caught this automatically.
- **Round 2 fix:** start candidates at `2`, use `range(2, n + 1)`, add the `n < 2` guard,
  and trial-divide only up to `sqrt(num)` for efficiency.
- **Why this is ReACT:** each *Action* (generated code) was actually *run*, the *Observation*
  (test result / error) came from the real environment, and that observation drove the next
  *Thought* — reason → act → observe → fix, until the test passed.
